In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
# 1. Import library
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from supabase import create_client, Client
import os



In [2]:
# 2. Konfigurasi Supabase
TABLE_NAME = "tb_konsentrasi_gas"

COL_PM25 = "pm25_ugm3"
COL_NO= "no2_ugm3"
COL_PM10 = "pm10_ugm3"
COL_CO = "co_ugm3"
COL_SUHU = "temperature"
COL_KELEMBAPAN = "humidity"
COL_TIMESTAMP = "created_at"

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)


In [3]:
# 3. Tentukan rentang waktu (7 hari terakhir)
seven_days_ago = datetime.now() - timedelta(days=14)
print(f"Mengambil data sejak {seven_days_ago}")


Mengambil data sejak 2026-03-26 19:01:52.689747


In [4]:
# 4. Pagination
all_data = []
page_size = 1000
offset = 0

while True:
    print(f"Mengambil data ke-{offset} sampai {offset + page_size}...")
    
    response = supabase.table(TABLE_NAME) \
        .select(f"{COL_PM25},{COL_PM10}, {COL_NO}, {COL_CO}, {COL_SUHU}, {COL_KELEMBAPAN}, {COL_TIMESTAMP}") \
        .gte(COL_TIMESTAMP, seven_days_ago.isoformat()) \
        .order(COL_TIMESTAMP, desc=False) \
        .range(offset, offset + page_size - 1) \
        .execute()
    
    data_batch = response.data
    
    if not data_batch:
        print("Semua data berhasil diambil.")
        break
    
    all_data.extend(data_batch)
    offset += len(data_batch)


Mengambil data ke-0 sampai 1000...
Mengambil data ke-1000 sampai 2000...
Mengambil data ke-2000 sampai 3000...
Mengambil data ke-3000 sampai 4000...
Mengambil data ke-4000 sampai 5000...
Mengambil data ke-5000 sampai 6000...
Mengambil data ke-6000 sampai 7000...
Mengambil data ke-7000 sampai 8000...
Mengambil data ke-8000 sampai 9000...
Mengambil data ke-9000 sampai 10000...
Mengambil data ke-10000 sampai 11000...
Mengambil data ke-11000 sampai 12000...
Mengambil data ke-11639 sampai 12639...
Semua data berhasil diambil.


In [5]:
df_raw = pd.DataFrame(all_data)

if df_raw.empty:
    raise ValueError("Tidak ada data yang ditemukan.")
    
# Konversi numerik
for col in [COL_PM25,COL_PM10,COL_NO, COL_CO, COL_SUHU, COL_KELEMBAPAN]:
    df_raw[col] = pd.to_numeric(df_raw[col], errors="coerce")

# Konversi timestamp
df_raw[COL_TIMESTAMP] = pd.to_datetime(df_raw[COL_TIMESTAMP]).dt.tz_localize(None)

# Drop NA
df_raw.dropna(subset=[COL_PM25,COL_PM10,COL_NO, COL_CO, COL_SUHU, COL_KELEMBAPAN], inplace=True)

# Set index waktu
df_clean = df_raw.set_index(COL_TIMESTAMP)
df_clean = df_clean[~df_clean.index.duplicated(keep="first")]
df_clean.sort_index(inplace=True)

# Rename kolom
df_clean.rename(columns={
    COL_PM25: "pm2.5",
    COL_PM10: "pm10_ugm3",
    COL_NO: "no2_ugm3",
    COL_CO: "co_ugm3",
    COL_SUHU: "temperature",
    COL_KELEMBAPAN: "humidity"
}, inplace=True)

print(f"Total data bersih: {len(df_clean)}")
df_clean.head()


Total data bersih: 11639


,pm2.5,pm10_ugm3,no2_ugm3,co_ugm3,temperature,humidity
created_at,,,,,,
2026-03-30 12:25:56.915536,62.83,80.19,0.0,50685.73,30.4,92.1
2026-03-30 12:26:58.086286,60.54,76.70,0.0,55446.97,30.5,91.9
2026-03-30 12:27:59.619867,56.66,70.66,0.0,55446.97,30.6,91.5
2026-03-30 12:29:00.733497,72.70,52.54,0.0,57056.88,30.7,90.5
2026-03-30 12:30:01.936268,69.88,90.41,0.0,57981.73,30.7,90.1


In [6]:
# [code]
# 2. Rekayasa Fitur untuk Prediksi 1 Menit ke Depan

if 'df_clean' in locals() and not df_clean.empty:
    print("Langkah 2: Membuat fitur dari data per menit...")
    
    # --- PERUBAHAN UTAMA: TIDAK ADA RESAMPLING ---
    # Kita bekerja langsung dengan df_clean yang berindeks per menit
    features_df = df_clean.copy()
    feature_cols = ['pm2.5', "pm10_ugm3",
    "co_ugm3",
    "no2_ugm3",
    "temperature",
    "humidity"]
    
    # Buat fitur lag dan rolling yang sesuai untuk skala menit
    for col in feature_cols:
        # Fitur Lag (jeda waktu)
        features_df[f'{col}_lag_1min'] = features_df[col].shift(1)
        features_df[f'{col}_lag_5min'] = features_df[col].shift(5)
        features_df[f'{col}_lag_15min'] = features_df[col].shift(15)
        features_df[f'{col}_lag_60min'] = features_df[col].shift(60) # 1 jam lalu

        # Fitur Rolling Window (jendela geser)
        features_df[f'{col}_rolling_mean_5min'] = features_df[col].rolling(window=5).mean()
        features_df[f'{col}_rolling_std_5min'] = features_df[col].rolling(window=5).std()
        features_df[f'{col}_rolling_mean_15min'] = features_df[col].rolling(window=15).mean()

    # Fitur Berbasis Waktu (tambahkan 'minute')
    features_df['minute'] = features_df.index.minute
    features_df['hour'] = features_df.index.hour
    features_df['dayofweek'] = features_df.index.dayofweek

    # Tentukan Target (y): nilai di 1 menit ke depan
    features_df['target'] = features_df['pm2.5'].shift(-1)
    
    # Hapus baris dengan nilai NaN
    features_df.dropna(inplace=True)

    print(f"\nJumlah baris akhir setelah pembuatan fitur: {len(features_df)}")
    print(f"Jumlah total fitur: {len(features_df.columns)}")
    
    if features_df.empty:
        print("ERROR: DataFrame fitur kosong.")
    else:
        print("\n5 baris pertama dari DataFrame fitur:")
        print(features_df.head())
else:
    print("Langkah 2 dilewati.")

Langkah 2: Membuat fitur dari data per menit...

Jumlah baris akhir setelah pembuatan fitur: 11578
Jumlah total fitur: 52

5 baris pertama dari DataFrame fitur:
                             pm2.5  pm10_ugm3  no2_ugm3   co_ugm3  \
created_at                                                          
2026-03-30 13:26:59.234565  128.74     164.75       0.0  59283.37   
2026-03-30 13:28:00.052899   99.66     127.11       0.0  58118.74   
2026-03-30 13:29:01.134223  126.80     166.37       0.0  58872.32   
2026-03-30 13:30:02.176204  109.53     154.76       0.0  59351.87   
2026-03-30 13:31:02.891395  135.61     165.45       0.0  59077.84   

                            temperature  humidity  pm2.5_lag_1min  \
created_at                                                          
2026-03-30 13:26:59.234565         32.4      83.1          132.44   
2026-03-30 13:28:00.052899         32.4      83.9          128.74   
2026-03-30 13:29:01.134223         32.4      83.6           99.66   
2026-03-30

In [7]:
df_ml = features_df.copy()
X = df_ml

y = df_ml["pm2.5"]

split_idx = int(len(df_ml) * 0.8)

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]


In [8]:
from xgboost import XGBRegressor

model_xgb = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42
)

model_xgb.fit(X_train, y_train)


,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None
,feature_types,None


In [9]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_pred = model_xgb.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
r2   = r2_score(y_test, y_pred)

print("=== Evaluasi XGBoost ===")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAPE : {mape:.2f}%")
print(f"R²   : {r2:.4f}")


=== Evaluasi XGBoost ===
MAE  : 0.0554
RMSE : 0.0860
MAPE : 0.31%
R²   : 0.9984


In [10]:
import joblib
joblib.dump(model_xgb, '../ml_model/xgb_pm25.pkl')
print("✅ Model tersimpan ke: ml_model/xgb_pm25.pkl")


✅ Model tersimpan ke: ml_model/xgb_pm25.pkl
